In [1]:
import pandas as pd
import numpy as np

In [2]:
VPOP_FILE = ("VPOP_2022v1_full_annual_20240301_18jun2024_nf_v6.csv")
OUTPUT_FILE = ("VPOP_2022v1_full_annual_20240301_18jun2024_nf_v6_ZeroPre2010s.csv")
AGE_FILE = "cdb17031y2022moves_age_dist.csv"
BASE_YEAR = 2022

In [9]:
CMAP_FIPS = [
    "17031",  # Cook
    "17043",  # DuPage
    "17089",  # Kane
    "17093",  # Kendall
    "17097",  # Lake
    "17111",  # McHenry
    "17197",  # Will
]

TRUCK_SOURCE_TYPES = [52, 53, 61, 62]

activity_columns = [
    "country_cd",
    "region_cd",
    "tribal_code",
    "census_tract_cd",
    "shape_id",
    "scc",
    "CD",
    "MSR",
    "activity_type",
    "ann_parm_value",
    "calc_year",
    "date_updated",
    "data_set_id",
    "jan_value",
    "feb_value",
    "mar_value",
    "apr_value",
    "may_value",
    "jun_value",
    "jul_value",
    "aug_value",
    "sep_value",
    "oct_value",
    "nov_value",
    "dec_value",
    "comment",
]
value_columns = [
    "ann_parm_value",
    "jan_value",
    "feb_value",
    "mar_value",
    "apr_value",
    "may_value",
    "jun_value",
    "jul_value",
    "aug_value",
    "sep_value",
    "oct_value",
    "nov_value",
    "dec_value",
]


In [5]:
header_lines = []

with open(VPOP_FILE, "r") as f:
    for line in f:
        if line.startswith("#"):
            header_lines.append(line)
        else:
            break
            
vpop = pd.read_csv(VPOP_FILE, comment="#", header=None, names=activity_columns,
    dtype={"region_cd": str, "scc": str,},)

# MOVES source type is embedded in SCC positions 5–6
vpop["source_type"] = (vpop["scc"].str[4:6].astype(int))

In [6]:
age_df = pd.read_csv(AGE_FILE)
age_df = age_df[(age_df["yearID"] == BASE_YEAR) & age_df["sourceTypeID"].isin(TRUCK_SOURCE_TYPES)].copy()

# 2022 - ageID gives model year
age_df["modelYear"] = (BASE_YEAR - age_df["ageID"])

In [7]:
legacy_fraction = (age_df[age_df["modelYear"] < 2010].groupby("sourceTypeID")["ageFraction"].sum())

print("\n" + "=" * 75)
print("PRE-2010 VPOP FRACTIONS")
print("=" * 75)

for source_type in TRUCK_SOURCE_TYPES:
    frac = legacy_fraction.loc[source_type]
    print(f"Source type {source_type}: "f"{frac * 100:.2f}% legacy")


PRE-2010 VPOP FRACTIONS
Source type 52: 48.03% legacy
Source type 53: 49.38% legacy
Source type 61: 32.56% legacy
Source type 62: 47.22% legacy


In [10]:
vpop_original = vpop.copy()

for source_type in TRUCK_SOURCE_TYPES:
    keep_fraction = (1.0 - legacy_fraction.loc[source_type])
    mask = (vpop["region_cd"].isin(CMAP_FIPS) & vpop["activity_type"].eq("POP") & vpop["source_type"].eq(source_type))
    vpop.loc[mask, value_columns] = (vpop.loc[mask, value_columns] * keep_fraction)

In [11]:
before = (
    vpop_original[vpop_original["region_cd"].isin(CMAP_FIPS) & vpop_original["activity_type"].eq("POP")
        & vpop_original["source_type"].isin(TRUCK_SOURCE_TYPES)].groupby("source_type")["ann_parm_value"].sum())

after = (
    vpop[vpop["region_cd"].isin(CMAP_FIPS) & vpop["activity_type"].eq("POP")
        & vpop["source_type"].isin(TRUCK_SOURCE_TYPES)].groupby("source_type")["ann_parm_value"].sum())

validation = pd.DataFrame({"Original VPOP": before,"ZeroPre2010s VPOP": after,})
validation["Legacy VPOP Removed"] = (validation["Original VPOP"]- validation["ZeroPre2010s VPOP"])
validation["Removed %"] = (validation["Legacy VPOP Removed"]/ validation["Original VPOP"]* 100)

print("\n" + "=" * 75)
print("ZERO-PRE-2010 VPOP VALIDATION")
print("=" * 75)

print(
    validation.to_string(
        formatters={
            "Original VPOP": lambda x: f"{x:,.2f}",
            "ZeroPre2010s VPOP": lambda x: f"{x:,.2f}",
            "Legacy VPOP Removed": lambda x: f"{x:,.2f}",
            "Removed %": lambda x: f"{x:.2f}%",}))


ZERO-PRE-2010 VPOP VALIDATION
            Original VPOP ZeroPre2010s VPOP Legacy VPOP Removed Removed %
source_type                                                              
52             114,508.69         59,510.56           54,998.13    48.03%
53               5,158.15          2,610.88            2,547.27    49.38%
61              12,701.86          8,565.56            4,136.30    32.56%
62              19,061.87         10,060.31            9,001.56    47.22%


In [ ]:
original_cmap_total = (
    vpop_original[
        vpop_original["region_cd"].isin(CMAP_FIPS)
        & vpop_original["activity_type"].eq("POP")
    ]["ann_parm_value"]    .sum())

new_cmap_total = (vpop[vpop["region_cd"].isin(CMAP_FIPS)& vpop["activity_type"].eq("POP")
    ]["ann_parm_value"].sum())

removed_cmap_total = original_cmap_total - new_cmap_total

print("\n" + "=" * 75)
print("TOTAL CMAP VPOP")
print("=" * 75)

print(f"\nOriginal CMAP VPOP: {original_cmap_total:,.2f}")
print(f"ZeroPre2010s CMAP VPOP: {new_cmap_total:,.2f}")
print(f"Legacy trucks removed: {removed_cmap_total:,.2f}")
print(f"Share of total CMAP VPOP removed: " f"{removed_cmap_total / original_cmap_total * 100:.2f}%")
